# Drum-wise PCA + PC algorithm + T²

`config -> load + clean -> flare masking -> per-drum PCA -> PC algorithm -> graph -> T²`

Code marked **[TA]** is copied from `references/CodeByTA/flare_pca_sfa_monitor_4windows_v2.ipynb`
(the reference pipeline; it is not modified). Drum grouping comes from `unit_drum_grouping/drum_tags.json`
and `unit_drums.json`. Unit renames (ISOM2→LPG, ISOM4→DHT, NSU_NHT_CCR→CCR, APS_NVPS_LEU→CDU) are applied
in memory at load time; raw parquet files are only read.

## Config

In [ ]:
import glob
import json
import os
import re
import time
from functools import reduce
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import Image, display
from causallearn.search.ConstraintBased.PC import pc

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 50)

# ---- paths ----
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "Dataset_HPCL"
GROUPING_DIR = ROOT / "unit_drum_grouping"
OUT_DIR = Path.cwd() / "outputs"
UNIT_RENAME = {"ISOM_2": "LPG", "ISOM_4": "DHT", "NSU_NHT_CCR": "CCR", "APS_NVPS_LEU": "CDU"}  # folder -> unit

# ---- [TA] data / cleaning ----
FLARE_FOLDER = "flare_data"
SKIP_ROWS = 58677                  # [TA] rows dropped at the start of every folder
TS = "Timestamp"
FLOW = "Flare_Flowrate"
MAX_MISSING_PCT = 10               # [TA] clean_and_impute default (process tags)
FLARE_MAX_MISSING_PCT = 50         # [TA] value used for the flare folder
SLOPE_STD_THRESHOLD = 1e-10        # [TA]
SCREEN_MIN_STD, SCREEN_MAX_MISSING, SCREEN_MAX_FLAT = 1e-3, 0.05, 0.999   # [TA] build_unit_matrix

# ---- [TA] flare events ----
THRESHOLD = 1.5                    # MT/h, flow >= THRESHOLD
MIN_DURATION_MIN = 30              # sustained at least this long
MERGE_GAP_MIN = 0                  # 0 -> strictly continuous

# ---- this notebook ----
FLARE_PRE_WINDOW_MIN = 60          # minutes before a flare start that are also removed
PCA_VARIANCE_THRESHOLD = 0.80      # smallest k with cumulative explained variance >= this
PC_ALPHA = 0.05                    # Fisher-z significance level of the PC algorithm
PC_ROW_STEP = 10                   # PC algorithm uses every n-th normal-operation row
PC_MAX_K = 3                       # largest conditioning set (causal-learn max_k)
STD_FLOOR = 1e-4                   # [TA] sd <= this on normal data -> near-zero-variance tag, dropped
SD_REL_MIN = 0.1                   # [TA] sd floored at SD_REL_MIN x long-run sd
USE_Z_CLIP, Z_CLIP = False, 50.0   # [TA] optional |z| cap (off)
T2_DTYPE = "float32"               # storage precision of the saved T² files
PLOT_RESAMPLE = "60min"            # T² plots show the max per bin
N_EXAMPLE_PLOTS = 3                # figures shown inline (all are saved)
EPS = 1e-12

for sub in ("graphs_json", "graphs_png", "t2", "t2_plots"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

## Step 0 - Inspect the flare data and the sampling interval

In [ ]:
rows = []
for f in sorted((DATA_DIR / FLARE_FOLDER).glob("*.parquet")):
    d = pd.read_parquet(f)
    t = d[TS]
    steps = t.diff().value_counts()
    rows.append({"file": f.name, "value_column": [c for c in d.columns if c != TS][0], "rows": len(d),
                 "ts_dtype": str(t.dtype), "tz": str(t.dt.tz), "start": t.min(), "end": t.max(),
                 "main_step": steps.index[0], "duplicate_ts": int(t.duplicated().sum()),
                 "gaps_>1min": int((t.diff() > pd.Timedelta(minutes=1)).sum()),
                 "largest_gap": t.diff().max(), "min": d.iloc[:, 1].min(), "max": d.iloc[:, 1].max()})
flare_files = pd.DataFrame(rows)
print("Flare folder:")
display(flare_files)
print("Flare flow used (as in the TA notebook): Flare_Flowrate = 509fi5201.daca.pv/1000 "
      "+ (LRflare+LREflare+99fi4601)/1000 - (44ft2505*509mw5101/22.414)/1000  [MT/h]")
print("Coverage: one refinery-wide flare flow; nothing in the files ties a flare to a unit -> mask applied globally.")

# sampling interval of the process data (raw files, before the 1-min grid)
proc_files = sorted(f for f in DATA_DIR.glob("*/*.parquet") if f.parent.name != FLARE_FOLDER)
step_counts = pd.Series(dtype=float)
for f in proc_files[:: max(1, len(proc_files) // 25)]:
    step_counts = step_counts.add(pd.read_parquet(f, columns=[TS])[TS].diff().value_counts(), fill_value=0)
print(f"\nProcess data: {len(proc_files)} tag files. Sampling steps in a sample of "
      f"{len(proc_files[:: max(1, len(proc_files) // 25)])} files:")
print(step_counts.sort_values(ascending=False).head(5).astype(int).to_string())

## Step 1 - Load and clean (global, before any split by unit/drum)

In [ ]:
drum_tags = json.loads((GROUPING_DIR / "drum_tags.json").read_text())
unit_drums = json.loads((GROUPING_DIR / "unit_drums.json").read_text())
DRUM_UNIT = {d: u for u, ds in unit_drums.items() for d in ds}
cv_present = any(v.get("control_valve_tags") for v in drum_tags.values())
print(f"{len(unit_drums)} units, {len(drum_tags)} drums; control_valve_tags in drum_tags.json: "
      f"{'yes' if cv_present else 'none (key absent or empty) -> only process tags are used'}")

FOLDER_OF = {UNIT_RENAME.get(p.name, p.name): p.name for p in DATA_DIR.iterdir()
             if p.is_dir() and p.name != FLARE_FOLDER}
missing_units = set(unit_drums) - set(FOLDER_OF)
assert not missing_units, f"no folder for units {missing_units}"
print("unit -> folder:", {u: FOLDER_OF[u] for u in unit_drums})


# [TA] -------------------------------------------------------------------------
def load_parquet_folder(folder, freq="1min"):
    """All parquet files of one folder -> one DataFrame on a complete time grid."""
    files = glob.glob(os.path.join(folder, "*.parquet"))
    if not files:
        raise ValueError(f"No parquet files found in '{folder}'")
    dfs = []
    for file in files:
        df = pd.read_parquet(file)
        df[TS] = pd.to_datetime(df[TS])
        df = df.drop_duplicates(subset=TS).set_index(TS)
        tag = os.path.splitext(os.path.basename(file))[0]
        dfs.append(df[[df.columns[0]]].rename(columns={df.columns[0]: tag}))
    merged = reduce(lambda a, b: a.join(b, how="outer"), dfs)
    grid = pd.date_range(merged.index.min(), merged.index.max(), freq=freq)
    merged = merged.reindex(grid).rename_axis(TS).reset_index()
    return merged.iloc[SKIP_ROWS:].reset_index(drop=True)


def clean_and_impute(df, max_missing_pct=10, slope_std_threshold=1e-10):
    ts = df[TS]
    x = df.drop(columns=[TS])
    before = x.shape[1]
    x = x.loc[:, x.isna().mean() * 100 <= max_missing_pct]     # gappy
    x = x.loc[:, x.std() > 0]                                  # constant
    x = x.loc[:, x.diff().std() > slope_std_threshold]         # constant slope
    x = x.loc[:, ~(x < 0).any()]                               # negative values
    x = x.interpolate(method="linear", limit_direction="both").ffill().bfill()
    print(f"  {before} tags -> {x.shape[1]} kept")
    return pd.concat([ts.reset_index(drop=True), x.reset_index(drop=True)], axis=1)
# -------------------------------------------------------------------------------


t0 = time.time()
parts = []
for u in unit_drums:                         # unit renamed in memory: column "UNIT::tag"
    d = load_parquet_folder(str(DATA_DIR / FOLDER_OF[u])).set_index(TS)
    d.columns = [f"{u}::{c}" for c in d.columns]
    parts.append(d)
raw_all = pd.concat(parts, axis=1).sort_index().rename_axis(TS).reset_index()   # one global frame
del parts
flare_raw = load_parquet_folder(str(DATA_DIR / FLARE_FOLDER))
print(f"loaded in {time.time() - t0:.0f} s")
print(f"BEFORE cleaning: {raw_all.shape[0]:,} rows x {raw_all.shape[1] - 1} tags "
      f"({raw_all[TS].iloc[0]} -> {raw_all[TS].iloc[-1]}), missing cells {raw_all.isna().mean().mean():.2%}")

In [ ]:
# [TA] cleaning, applied once to the global frame
print("process", end="")
clean_all = clean_and_impute(raw_all, max_missing_pct=MAX_MISSING_PCT, slope_std_threshold=SLOPE_STD_THRESHOLD)
dropped_clean = sorted(set(raw_all.columns) - set(clean_all.columns))
del raw_all


# [TA] build_unit_matrix screening, same checks, run on the global frame (columns already "UNIT::tag")
def screen_matrix(df, min_std=SCREEN_MIN_STD, max_missing=SCREEN_MAX_MISSING, max_flat=SCREEN_MAX_FLAT):
    merged = df.drop_duplicates(subset=TS).set_index(TS).select_dtypes("number")
    keep, seen, dropped = [], {}, []
    for c in merged.columns:
        v = merged[c]
        vv = v.dropna()
        why = None
        if v.isna().mean() > max_missing:
            why = "too many missing"
        elif vv.nunique() <= 1 or vv.std() <= min_std:
            why = "stuck / constant"
        elif (vv.diff().fillna(0) == 0).mean() > max_flat:
            why = "flat"
        else:
            h = np.ascontiguousarray(v.to_numpy(float)).tobytes()
            if h in seen:
                why = f"duplicate of {seen[h]}"
            seen.setdefault(h, c)
        if why:
            dropped.append(f"{c} ({why})")
        else:
            keep.append(c)
    if dropped:
        print(f"screened out {len(dropped)} tags: " + ", ".join(dropped))
    return merged[keep].dropna().sort_index(), dropped


data, dropped_screen = screen_matrix(clean_all)
del clean_all
print(f"AFTER cleaning: {data.shape[0]:,} rows x {data.shape[1]} tags; "
      f"dropped by clean_and_impute: {len(dropped_clean)}, by screening: {len(dropped_screen)}")
if dropped_clean:
    print("  clean_and_impute dropped:", dropped_clean)

print("flare", end="")
flare_df = clean_and_impute(flare_raw, max_missing_pct=FLARE_MAX_MISSING_PCT)
flare_df[FLOW] = (flare_df["509fi5201.daca.pv_1000"]                                      # [TA]
                  + flare_df["(LRflare.pv+LREflare.pv+99fi4601.daca.pv)_1000"]
                  - flare_df["(44ft2505.pv_509mw5101.daca.pv_22.414)_1000"])
print(f"flare data {flare_df[TS].iloc[0]} to {flare_df[TS].iloc[-1]}, {len(flare_df):,} rows")

## Step 2 - Remove flare + pre-flare window
Flare events use the TA definition (flow ≥ `THRESHOLD` for ≥ `MIN_DURATION_MIN`). A row is removed if it lies
inside an event [start, end] or in [start − `FLARE_PRE_WINDOW_MIN`, start). The timestamp index is kept.

In [ ]:
# [TA] -------------------------------------------------------------------------
def runs(mask):
    """(start, end) index pairs of the True runs in a boolean array."""
    d = np.diff(np.r_[0, mask.astype(np.int8), 0])
    return list(zip(np.flatnonzero(d == 1), np.flatnonzero(d == -1)))


def label_flare_events(df):
    """Flare event = flow >= THRESHOLD for >= MIN_DURATION_MIN. Runs separated
    by <= MERGE_GAP_MIN minutes below the threshold are merged first
    (MERGE_GAP_MIN = 0 -> strictly continuous)."""
    d = df[[TS, FLOW]].drop_duplicates(TS).sort_values(TS)
    step = float(d[TS].diff().dt.total_seconds().median() / 60)
    grid = pd.date_range(d[TS].iloc[0], d[TS].iloc[-1], freq=pd.Timedelta(minutes=step))
    d = d.set_index(TS).reindex(grid).rename_axis(TS).reset_index()   # gaps break a run
    above = (d[FLOW] >= THRESHOLD).to_numpy().copy()
    gap = int(round(MERGE_GAP_MIN / step))
    if gap > 0:                          # join runs split by a short dip / gap
        r = runs(above)
        for (_, e0), (s1, _) in zip(r[:-1], r[1:]):
            if s1 - e0 <= gap:
                above[e0:s1] = True
    rows = []
    for s, e in runs(above):
        if e - s < np.ceil(MIN_DURATION_MIN / step):
            continue
        seg = d.iloc[s:e]
        rows.append({"event_id": len(rows) + 1,
                     "start": seg[TS].iloc[0], "end": seg[TS].iloc[-1],
                     "duration_min": (e - s) * step,
                     "peak_flow": float(seg[FLOW].max()),
                     "volume_MT": float(seg[FLOW].sum() * step / 60)})
    return pd.DataFrame(rows)
# -------------------------------------------------------------------------------


events = label_flare_events(flare_df)
ts = data.index
in_flare = np.zeros(len(ts), bool)
in_pre = np.zeros(len(ts), bool)
for e in events.itertuples(index=False):
    in_flare |= (ts >= e.start) & (ts <= e.end)
    in_pre |= (ts >= e.start - pd.Timedelta(minutes=FLARE_PRE_WINDOW_MIN)) & (ts < e.start)
pre_only = in_pre & ~in_flare
removed = in_flare | in_pre
normal = data.loc[~removed]                       # original timestamps kept as the index

ev_in = events[(events.end >= ts[0]) & (events.start <= ts[-1])]
print(f"flare events: {len(events)} in the flare record, {len(ev_in)} overlap the cleaned process data")
print(f"total rows          {len(ts):>9,}")
print(f"removed, flare      {in_flare.sum():>9,}")
print(f"removed, pre-window {pre_only.sum():>9,}  (rows in a pre-window that are not inside a flare)")
print(f"rows remaining      {len(normal):>9,}  ({len(normal) / len(ts):.1%})")
display(events.head())
pd.DataFrame({"in_flare": in_flare, "in_pre_window": pre_only}, index=ts).to_parquet(OUT_DIR / "flare_mask.parquet")

## Step 3 - Drum-wise PCA
Per drum: tags present after cleaning → drop near-zero-variance tags on normal data (sd ≤ `STD_FLOOR`) →
standardise with normal-data mean/sd ([TA]: sd floored at `SD_REL_MIN` × long-run sd) → [TA] `pca_block`
with the variance target set to `PCA_VARIANCE_THRESHOLD`.

In [ ]:
# [TA] pca_block, variance target taken from PCA_VARIANCE_THRESHOLD ------------
VAR_TARGET = 100 * PCA_VARIANCE_THRESHOLD


def pca_block(Z):
    """PCA of one unit's standardised block: loadings for VAR_TARGET % variance."""
    if Z.shape[1] == 1:
        return np.ones((1, 1))
    lam, V = np.linalg.eigh(np.cov(Z, rowvar=False))
    o = np.argsort(lam)[::-1]
    lam, V = np.clip(lam[o], 0, None), V[:, o]
    if lam.sum() <= EPS:
        return np.eye(Z.shape[1])[:, :1]
    k = int(np.argmax(100 * np.cumsum(lam) / lam.sum() >= VAR_TARGET - 1e-10) + 1)
    return V[:, :k]
# -------------------------------------------------------------------------------


def safe(name):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", name).strip("_")


sd_long = data.std(ddof=1)                        # [TA] long-run sd (whole cleaned record)
MODELS, pca_rows, file_names = {}, [], set()
for u, drums in unit_drums.items():
    for d in drums:
        tags = drum_tags[d]["process_tags"]
        cols = [f"{u}::{t}" for t in tags]
        present = [c for c in cols if c in data.columns]
        lost = [c.split("::", 1)[1] for c in cols if c not in data.columns]
        sd_w = normal[present].std(ddof=1)
        active = [c for c in present if np.isfinite(sd_w[c]) and sd_w[c] > STD_FLOOR]
        nzv = [c.split("::", 1)[1] for c in present if c not in active]
        row = {"unit": u, "drum": d, "n_tags_json": len(tags), "removed_by_cleaning": len(lost),
               "near_zero_var": len(nzv), "n_tags": len(active), "n_PCs": 0, "cum_var": np.nan,
               "flag": "", "dropped_tags": ", ".join(lost + nzv)}
        if not active:
            row["flag"] = "SKIPPED: no usable tag"
            pca_rows.append(row)
            continue
        X = normal[active].to_numpy(float)
        mu, sd = X.mean(0), X.std(0, ddof=1)
        sd = np.maximum(sd, SD_REL_MIN * np.nan_to_num(sd_long[active].to_numpy()))   # [TA] floor
        Z = (X - mu) / sd
        if USE_Z_CLIP:
            Z = np.clip(Z, -Z_CLIP, Z_CLIP)
        P = pca_block(Z)
        lam = np.clip(np.sort(np.linalg.eigvalsh(np.atleast_2d(np.cov(Z, rowvar=False))))[::-1], 0, None)
        k = P.shape[1]
        Y = Z @ P
        stem = f"{u}__{safe(d)}"
        assert stem not in file_names, f"file name clash {stem}"
        file_names.add(stem)
        MODELS[d] = {"unit": u, "cols": active, "mu": mu, "sd": sd, "P": P,
                     "mu_y": Y.mean(0), "var_y": Y.var(0, ddof=1), "lam": lam, "stem": stem,
                     "nodes": [f"{d}_PC{j + 1}" for j in range(k)]}
        flags = []
        if len(active) < 2:
            flags.append("<2 tags")
        if nzv:
            flags.append(f"{len(nzv)} near-zero-var tag(s) dropped")
        if lost:
            flags.append(f"{len(lost)} tag(s) removed by cleaning")
        row.update(n_PCs=k, cum_var=float(lam[:k].sum() / max(lam.sum(), EPS)), flag="; ".join(flags))
        pca_rows.append(row)

pca_table = pd.DataFrame(pca_rows)
pca_table.to_csv(OUT_DIR / "drum_pca_summary.csv", index=False)
with pd.option_context("display.max_rows", 400):
    display(pca_table.drop(columns="dropped_tags").round({"cum_var": 3}))
print(f"drums modelled: {len(MODELS)} of {len(pca_table)}; PCs in total: {sum(m['P'].shape[1] for m in MODELS.values())}")
print("PCs kept per drum:", pca_table.n_PCs.value_counts().sort_index().to_dict())
print("drums with <2 tags:", int((pca_table.n_tags < 2).sum()))
print("near-zero-variance tags dropped:", int(pca_table.near_zero_var.sum()),
      "| tags removed by cleaning (in JSON, not in cleaned data):", int(pca_table.removed_by_cleaning.sum()))
print("control valve tags: none in drum_tags.json, so none used")

## Step 4 - PC algorithm per drum
causal-learn `pc` (Fisher-z, `alpha = PC_ALPHA`, stable) on the retained PC scores of the normal-operation rows
(every `PC_ROW_STEP`-th row). In causal-learn `G[i,j] = -1, G[j,i] = 1` means i → j; both −1 means i — j;
both 1 means i ↔ j.

In [ ]:
def edge_list(G, names):
    """CPDAG matrix from causal-learn -> list of {source, target, type}."""
    out = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = G[i, j], G[j, i]
            if a == 0 and b == 0:
                continue
            if a == -1 and b == 1:
                out.append({"source": names[i], "target": names[j], "type": "->"})
            elif a == 1 and b == -1:
                out.append({"source": names[j], "target": names[i], "type": "->"})
            elif a == -1 and b == -1:
                out.append({"source": names[i], "target": names[j], "type": "--"})
            elif a == 1 and b == 1:
                out.append({"source": names[i], "target": names[j], "type": "<->"})
            else:
                out.append({"source": names[i], "target": names[j], "type": f"other({a},{b})"})
    return out


GRAPHS, pc_rows = {}, []
t0 = time.time()
for d, m in MODELS.items():
    k = m["P"].shape[1]
    Zn = (normal[m["cols"]].to_numpy(float)[::PC_ROW_STEP] - m["mu"]) / m["sd"]
    D = Zn @ m["P"]
    if k == 1:
        G = np.zeros((1, 1), int)
        note = "trivial (1 PC)"
    else:
        cg = pc(np.ascontiguousarray(D), alpha=PC_ALPHA, indep_test="fisherz", stable=True, uc_rule=0,
                uc_priority=2, max_k=PC_MAX_K, verbose=False, show_progress=False)
        G = np.array(cg.G.graph, dtype=int)
        note = ""
    edges = edge_list(G, m["nodes"])
    corr = np.corrcoef(D, rowvar=False) if k > 1 else np.ones((1, 1))
    max_r = float(np.abs(corr - np.eye(k)).max()) if k > 1 else np.nan
    GRAPHS[d] = {"unit": m["unit"], "drum": d, "nodes": m["nodes"], "adjacency": G.tolist(), "edges": edges,
                 "n_rows": int(len(D)), "alpha": PC_ALPHA, "test": "fisherz", "max_k": PC_MAX_K,
                 "max_abs_corr_between_PCs": max_r}
    (OUT_DIR / "graphs_json" / f"{m['stem']}.json").write_text(json.dumps(GRAPHS[d], indent=1))
    pc_rows.append({"unit": m["unit"], "drum": d, "n_PCs": k, "n_edges": len(edges),
                    "directed": sum(e["type"] == "->" for e in edges),
                    "undirected": sum(e["type"] == "--" for e in edges),
                    "max_abs_corr_between_PCs": max_r, "note": note})
pc_table = pd.DataFrame(pc_rows)
pc_table.to_csv(OUT_DIR / "drum_pc_graph_summary.csv", index=False)
print(f"PC algorithm on {len(MODELS)} drums in {time.time() - t0:.0f} s, "
      f"{next(iter(GRAPHS.values()))['n_rows']:,} rows each (every {PC_ROW_STEP}th normal row)")
print(f"trivial graphs (only 1 PC kept): {int((pc_table.n_PCs == 1).sum())} drums")
multi = pc_table[pc_table.n_PCs > 1]
print(f"drums with >=2 PCs: {len(multi)}; with at least one edge: {int((multi.n_edges > 0).sum())}; "
      f"edges in total: {int(multi.n_edges.sum())}")
print(f"largest |correlation| between two PCs of the same drum (on the PC rows): {multi.max_abs_corr_between_PCs.max():.2e}")
with pd.option_context("display.max_rows", 400):
    display(multi.round({"max_abs_corr_between_PCs": 6}))

## Step 5 - Graph figures

In [ ]:
STYLE = {"->": dict(style="solid", arrows=True, arrowstyle="-|>"),
         "--": dict(style="solid", arrows=False),
         "<->": dict(style="dashed", arrows=True, arrowstyle="<|-|>")}


def draw_graph(ax, g, title=None, font=8):
    nodes = g["nodes"]
    Gx = nx.DiGraph()
    Gx.add_nodes_from(nodes)
    pos = nx.circular_layout(Gx) if len(nodes) > 1 else {nodes[0]: np.zeros(2)}
    nx.draw_networkx_nodes(Gx, pos, ax=ax, node_color="#cfe2f3", edgecolors="#1f4e79", node_size=900)
    nx.draw_networkx_labels(Gx, pos, ax=ax, labels={n: n.replace("_PC", "\n_PC") for n in nodes}, font_size=font)
    for e in g["edges"]:
        st = STYLE.get(e["type"], dict(style="dotted", arrows=False))
        kw = {k: v for k, v in st.items() if k != "style"}
        nx.draw_networkx_edges(Gx, pos, edgelist=[(e["source"], e["target"])], ax=ax, style=st["style"],
                               node_size=900, width=1.6, edge_color="#333333", **kw)
    ax.set_title(title or g["drum"], fontsize=font + 1)
    ax.margins(0.25)
    ax.axis("off")


matplotlib.use("Agg")
t0 = time.time()
for d, g in GRAPHS.items():
    m = MODELS[d]
    folder = OUT_DIR / "graphs_png" / g["unit"]
    folder.mkdir(exist_ok=True)
    fig, ax = plt.subplots(figsize=(4.5, 4))
    draw_graph(ax, g, f"{g['unit']} | {d}\n{len(g['edges'])} edge(s), {len(g['nodes'])} PC(s)")
    fig.tight_layout()
    fig.savefig(folder / f"{m['stem']}.png", dpi=120)
    plt.close(fig)

for u, drums in unit_drums.items():               # one summary figure per unit
    gs = [GRAPHS[d] for d in drums if d in GRAPHS]
    n = len(gs)
    ncol = min(6, n)
    nrow = int(np.ceil(n / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.0 * nrow), squeeze=False)
    for ax, g in zip(axs.flat, gs):
        draw_graph(ax, g, font=6)
    for ax in list(axs.flat)[n:]:
        ax.axis("off")
    fig.suptitle(f"{u}: PC-algorithm graphs per drum (nodes = drum_PC#; -> directed, — undirected)", fontsize=12)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "graphs_png" / f"_summary_{u}.png", dpi=110)
    plt.close(fig)
print(f"graph figures written in {time.time() - t0:.0f} s")
for u in list(unit_drums)[:2]:
    display(Image(filename=str(OUT_DIR / "graphs_png" / f"_summary_{u}.png")))

## Step 6 - T² and individual T²
[TA] `Model.statistics`: `S = (Z @ P - mu_y) @ W`, `T2_slow = Σ S[:, :M]²`, `T2_fast = Σ S[:, M:]²`, where
`W = Wh @ R` (`Wh` whitens the PCA scores, `R` is the SFA rotation, orthogonal). Without the SFA split this is
**T² = T2_slow + T2_fast = Σ_j ((y_j − μ_j)² / λ_j)** and the **individual T²** of PC *j* is its term
**T²_j = (y_j − μ_j)² / λ_j** (μ_j, λ_j = mean and variance of the PC score on normal-operation data). The cell
below checks this equality against the TA `sfa()` + `statistics()` code for every drum.
Computed on the **full cleaned timeline** (flare and pre-flare rows included), as the TA notebook scores every sample.

In [ ]:
# [TA] sfa + Model.statistics, used only to verify T2 == T2_slow + T2_fast --------
SFA_RCOND, SLOW_FRAC, MIN_SLOW, MIN_FAST = 1e-4, 0.2, 2, 2


def sfa(Y):
    mu_y = Y.mean(0)
    Yc = Y - mu_y
    lam, U = np.linalg.eigh(np.atleast_2d(np.cov(Yc, rowvar=False)))
    o = np.argsort(lam)[::-1]
    lam, U = lam[o], U[:, o]
    keep = lam > max(lam[0] * SFA_RCOND, EPS)
    Wh = U[:, keep] / np.sqrt(lam[keep])
    sp, R = np.linalg.eigh(np.atleast_2d(np.cov(np.diff(Yc @ Wh, axis=0), rowvar=False)))
    o = np.argsort(sp)
    return Wh @ R[:, o], np.clip(sp[o], EPS, None), mu_y


def ta_statistics(Z, P, W, mu_y, M):
    S = (Z @ P - mu_y) @ W
    return (S[:, :M] ** 2).sum(1), (S[:, M:] ** 2).sum(1)
# -------------------------------------------------------------------------------


def drum_t2(m, X):
    """T2 and individual T2 (one column per PC) for raw rows X of the drum's tags."""
    Z = (X - m["mu"]) / m["sd"]
    if USE_Z_CLIP:
        Z = np.clip(Z, -Z_CLIP, Z_CLIP)
    T2j = (Z @ m["P"] - m["mu_y"]) ** 2 / m["var_y"]
    return T2j.sum(1), T2j


# verification on the normal rows
worst = 0.0
for d, m in MODELS.items():
    X = normal[m["cols"]].to_numpy(float)
    Z = (X - m["mu"]) / m["sd"]
    W, _, mu_y = sfa(Z @ m["P"])
    r = W.shape[1]
    lo = max(1, MIN_SLOW)
    M = int(np.clip(np.ceil(SLOW_FRAC * r), lo, min(max(lo, r - MIN_FAST), max(1, r - 1))))
    t_s, t_f = ta_statistics(Z, m["P"], W, mu_y, M)
    t2, _ = drum_t2(m, X)
    worst = max(worst, float(np.max(np.abs(t2 - (t_s + t_f)) / np.maximum(t2, 1))))
print(f"max relative difference |T2 - (T2_slow + T2_fast)| over all drums: {worst:.2e}")

In [ ]:
Xall = data
t0 = time.time()
t2_index = []
for d, m in MODELS.items():
    t2, T2j = drum_t2(m, Xall[m["cols"]].to_numpy(float))
    out = pd.DataFrame(T2j.astype(T2_DTYPE), index=Xall.index, columns=[f"T2_{n}" for n in m["nodes"]])
    out.insert(0, "T2", t2.astype(T2_DTYPE))
    folder = OUT_DIR / "t2" / m["unit"]
    folder.mkdir(exist_ok=True)
    out.to_parquet(folder / f"{m['stem']}.parquet")
    t2_index.append({"unit": m["unit"], "drum": d, "file": str((folder / f"{m['stem']}.parquet").relative_to(OUT_DIR)),
                     "n_PCs": len(m["nodes"]), "T2_median_normal": float(np.median(t2[~removed])),
                     "T2_median_flare": float(np.median(t2[in_flare])) if in_flare.any() else np.nan,
                     "T2_median_prewindow": float(np.median(t2[pre_only])) if pre_only.any() else np.nan})
t2_table = pd.DataFrame(t2_index)
t2_table.to_csv(OUT_DIR / "t2_index.csv", index=False)
print(f"T2 files written for {len(t2_table)} drums in {time.time() - t0:.0f} s "
      f"({sum(f.stat().st_size for f in (OUT_DIR / 't2').rglob('*.parquet')) / 1e9:.2f} GB)")
display(t2_table.head(10).round(2))

In [ ]:
# T2 over time per drum (max per PLOT_RESAMPLE bin), flare events red, pre-flare windows orange
spans = [(e.start - pd.Timedelta(minutes=FLARE_PRE_WINDOW_MIN), e.start, e.end) for e in ev_in.itertuples()]
t0 = time.time()
for i, r in enumerate(t2_table.itertuples()):
    s = pd.read_parquet(OUT_DIR / r.file, columns=["T2"])["T2"].resample(PLOT_RESAMPLE).max()
    fig, ax = plt.subplots(figsize=(14, 3.2))
    for w0, st, en in spans:
        ax.axvspan(w0, st, color="orange", alpha=0.25, lw=0)
        ax.axvspan(st, en, color="red", alpha=0.25, lw=0)
    ax.plot(s.index, s.values, lw=0.6, color="#1f4e79")
    ax.set_yscale("log")
    ax.set_ylabel("T²")
    ax.set_title(f"{r.unit} | {r.drum}  ({r.n_PCs} PC)  - max per {PLOT_RESAMPLE}; red = flare, "
                 f"orange = {FLARE_PRE_WINDOW_MIN}-min pre-flare window")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    folder = OUT_DIR / "t2_plots" / r.unit
    folder.mkdir(exist_ok=True)
    path = folder / f"{Path(r.file).stem}.png"
    fig.savefig(path, dpi=110)
    plt.close(fig)
    if i < N_EXAMPLE_PLOTS:
        display(Image(filename=str(path)))
print(f"T2 plots written in {time.time() - t0:.0f} s")

## Summary of outputs

In [ ]:
skipped = pca_table[pca_table.n_PCs == 0]
print(f"units: {len(unit_drums)} | drums in JSON: {len(drum_tags)} | drums modelled: {len(MODELS)} | skipped: {len(skipped)}")
if len(skipped):
    display(skipped[["unit", "drum", "flag", "dropped_tags"]])
print(f"trivial graphs (1 PC): {int((pc_table.n_PCs == 1).sum())} | graphs with >=2 PCs: {int((pc_table.n_PCs > 1).sum())} "
      f"| graphs with edges: {int((pc_table.n_edges > 0).sum())}")
for p in ["flare_mask.parquet", "drum_pca_summary.csv", "drum_pc_graph_summary.csv", "t2_index.csv"]:
    print(f"  outputs/{p}")
for sub in ["graphs_json", "graphs_png", "t2", "t2_plots"]:
    files = list((OUT_DIR / sub).rglob("*.*"))
    print(f"  outputs/{sub}/  {len(files)} files")